In [1]:
# transformers v5: restore head masks + fix attention mask API for IDEA Grounding DINO
from pathlib import Path
import sys

_pr = Path(".").resolve()
if str(_pr) not in sys.path:
    sys.path.insert(0, str(_pr))
import groundingdino_transformers_compat
groundingdino_transformers_compat.apply()

import os

# Antes de cargar el modelo: fija la GPU (solo afecta si se ejecuta antes de importar CUDA)
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import pandas as pd
from tqdm import tqdm
from autodistill_grounded_sam import GroundedSAM
from autodistill.detection import CaptionOntology

DATASET_PATH = "./datasets/sFERNANDO_TUGBOATS_FLUXDEV"
df = pd.read_csv(f"{DATASET_PATH}/prompts.csv")
df = df[df["class"] == 'tugboat']

# Orden fijo de clases en los .txt (YOLO): una entrada por clase del dataset
ALL_CLASSES = sorted(df["class"].unique())
CLASS_TO_ID = {c: i for i, c in enumerate(ALL_CLASSES)}

# Ontología dummy; en el bucle se asigna la ontología por imagen
model = GroundedSAM(ontology=CaptionOntology({"init": "init"}))

print(f"Clases ({len(ALL_CLASSES)}): {ALL_CLASSES}")
print(f"Filas / imágenes: {len(df)}")

/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


trying to load grounding dino directly


torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)


final text_encoder_type: bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Clases (1): ['tugboat']
Filas / imágenes: 768


Por cada fila se usa **`row["class"]`** como clase YOLO en el `.txt` (índice según `ALL_CLASSES`). El texto que ve Grounding DINO es `grounding_prompt_for(clase)` (p. ej. varias clases de barco → `"boat"`). No uses una lista fija tipo `ONTOLOGY_CLASSES = ["car"]`: con un solo prompt, `class_id` siempre es `0` y todas las cajas se etiquetaban mal.

In [2]:
classes_dict ={}
for x in ALL_CLASSES:
    classes_dict[x] = x


classes_dict


# Clase que guardas en el label → texto que usa GroundingDINO (p. ej. "person" detecta mejor)
GROUNDING_PROMPT_BY_CLASS = {
    "pedestrian": "person",
    "Person_sitting": "person",
    "person_sitting": "person",
    'cruise ship': 'boat',
 'fishing boat': 'boat',
 'human powered boat': 'boat',
 'industrial ship': 'boat',
 'jetski': 'boat',
 'military_ship': 'boat',
 'passenger ship': 'boat',
 'pedal boat': 'boat',
 'sailboat': 'boat',
 'small boat': 'boat',
 'speed_boat': 'boat',
 'tugboat': 'boat'
}

def grounding_prompt_for(label: str) -> str:
    return GROUNDING_PROMPT_BY_CLASS.get(label, GROUNDING_PROMPT_BY_CLASS.get(label.lower(), label))


In [ ]:
import cv2
import supervision as sv
import json
import numpy as np
import pandas as pd
from datetime import datetime

path = DATASET_PATH
output_path = os.path.join(path, "labels_multi")
labeled_images_dir = os.path.join(path, "images_multi")
json_dir = os.path.join(path, "detections_json")
os.makedirs(output_path, exist_ok=True)
os.makedirs(labeled_images_dir, exist_ok=True)
os.makedirs(json_dir, exist_ok=True)

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

# Prefer explicit image id from CSV (`Unnamed: 0`) if present — matches `images/{id}.png`
_ID_COL = "Unnamed: 0"


def row_image_id(index, row) -> int:
    if _ID_COL in row.index:
        return int(row[_ID_COL])
    return int(index)


for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    try:
        img_id = row_image_id(index, row)
        gt_class = str(row["class"])
        grounding_prompt = grounding_prompt_for(gt_class)

        # prompt sent to Grounding DINO → canonical class stored on detections
        model.ontology = CaptionOntology({grounding_prompt: gt_class})

        image_path = os.path.join(path, "images", f"{img_id}.png")
        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"No se pudo leer: {image_path}")

        h, w = image.shape[:2]
        txt_path = os.path.join(output_path, f"{img_id}.txt")

        detections = model.predict(image_path)
        detections = detections.with_nms()

        yolo_class_id = CLASS_TO_ID[gt_class]

        record = {
            "index": img_id,
            "dataframe_iter_index": int(index) if isinstance(index, (int, np.integer)) else index,
            "image_path": image_path,
            "image_width": w,
            "image_height": h,
            "ground_truth_class": gt_class,
            "grounding_prompt": grounding_prompt,
            "timestamp": datetime.now().isoformat(),
            "num_detections": len(detections) if detections is not None else 0,
            "detections": [],
        }

        # One GT class per image row; every box is written with CLASS_TO_ID[gt_class]
        if len(detections) > 0 and detections.xyxy is not None and len(detections.xyxy) > 0:
            confidences = detections.confidence if detections.confidence is not None else np.zeros(len(detections.xyxy))
            vis_labels = [f"{gt_class} {float(conf):.2f}" for conf in confidences]

            for i, (box, conf) in enumerate(zip(detections.xyxy, confidences)):
                x_min, y_min, x_max, y_max = box
                x_center = ((x_min + x_max) / 2) / w
                y_center = ((y_min + y_max) / 2) / h
                bw = (x_max - x_min) / w
                bh = (y_max - y_min) / h

                record["detections"].append({
                    "detection_id": i,
                    "class_name": gt_class,
                    "yolo_class_id": yolo_class_id,
                    "confidence": float(conf),
                    "bbox_xyxy": [float(x_min), float(y_min), float(x_max), float(y_max)],
                    "bbox_yolo": [float(x_center), float(y_center), float(bw), float(bh)],
                    "bbox_width_px": float(x_max - x_min),
                    "bbox_height_px": float(y_max - y_min),
                })

            with open(os.path.join(json_dir, f"{img_id}.json"), "w") as jf:
                json.dump(record, jf, indent=2)

            annotated_image = box_annotator.annotate(scene=image.copy(), detections=detections)
            annotated_image = label_annotator.annotate(
                scene=annotated_image,
                detections=detections,
                labels=vis_labels,
            )

            with open(txt_path, "w") as f:
                for box in detections.xyxy:
                    x_min, y_min, x_max, y_max = box
                    x_center = ((x_min + x_max) / 2) / w
                    y_center = ((y_min + y_max) / 2) / h
                    bw = (x_max - x_min) / w
                    bh = (y_max - y_min) / h
                    f.write(f"{yolo_class_id} {x_center} {y_center} {bw} {bh}\n")
        else:
            with open(os.path.join(json_dir, f"{img_id}.json"), "w") as jf:
                json.dump(record, jf, indent=2)
            annotated_image = image.copy()
            open(txt_path, "w").close()

        cv2.imwrite(os.path.join(labeled_images_dir, f"{img_id}.png"), annotated_image)

    except Exception as err:
        try:
            _iid = row_image_id(index, row)
        except Exception:
            _iid = None
        print(f"Error index={index} img_id={_iid}: {err}")


Processing:   0%|          | 0/768 [00:00<?, ?it/s]`use_return_dict` is deprecated! Use `return_dict` instead!
torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
None of the inputs have requires_grad=True. Gradients will be None
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
Processing:   0%|          | 1/768 [00:01<20:12,  1.58s/it]

Error index=0 img_id=0: CUDA out of memory. Tried to allocate 1024.00 MiB. GPU 0 has a total capacity of 23.56 GiB of which 572.06 MiB is free. Process 2939 has 17.32 GiB memory in use. Including non-PyTorch memory, this process has 5.65 GiB memory in use. Of the allocated memory 5.25 GiB is allocated by PyTorch, and 112.89 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


Processing:   0%|          | 2/768 [00:02<13:30,  1.06s/it]

Error index=1 img_id=1: CUDA out of memory. Tried to allocate 1024.00 MiB. GPU 0 has a total capacity of 23.56 GiB of which 642.06 MiB is free. Process 2939 has 17.32 GiB memory in use. Including non-PyTorch memory, this process has 5.58 GiB memory in use. Of the allocated memory 4.25 GiB is allocated by PyTorch, and 1.04 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


Processing:   0%|          | 3/768 [00:02<10:00,  1.27it/s]

...

Processing:   3%|▎         | 23/768 [00:15<08:03,  1.54it/s]

Error index=22 img_id=22: CUDA out of memory. Tried to allocate 1024.00 MiB. GPU 0 has a total capacity of 23.56 GiB of which 642.06 MiB is free. Process 2939 has 17.32 GiB memory in use. Including non-PyTorch memory, this process has 5.58 GiB memory in use. Of the allocated memory 4.25 GiB is allocated by PyTorch, and 1.04 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


Processing:   3%|▎         | 24/768 [00:15<07:48,  1.59it/s]

Error index=23 img_id=23: CUDA out of memory. Tried to allocate 1024.00 MiB. GPU 0 has a total capacity of 23.56 GiB of which 642.06 MiB is free. Process 2939 has 17.32 GiB memory in use. Including non-PyTorch memory, this process has 5.58 GiB memory in use. Of the allocated memory 4.25 GiB is allocated by PyTorch, and 1.04 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


Processing:   3%|▎         | 25/768 [00:16<07:04,  1.75it/s]